# Research Paper Summarizer

## What this project does

This notebook is my own implementation built on top of the concepts learned in AI Core Track - Ed Donner - Week1/day1.

The original day1 project scraped any website using `requests` + `BeautifulSoup` and summarized it with an LLM.
The problem: **research websites like ResearchGate block bots** and return 403 errors or security checkpoints.

### My solution
Instead of fighting bot-detection, I target **research platforms that provide free, open APIs**:

| Source | What it gives us |
|---|---|
| **arXiv API** | Title, authors, abstract, categories — no auth required |
| **Semantic Scholar API** | Same + citation count, references — no auth required |

We then feed the clean paper metadata to Claude (Anthropic) and get a structured, readable summary.

---

## Project structure

```
research_summarizer.ipynb
```

All code lives in this single notebook to keep things simple and self-contained.

---
## Step 1: Imports

These are the libraries we need:

- `os` + `dotenv` — to load our API key from the `.env` file
- `requests` — to make HTTP calls to the arXiv and Semantic Scholar APIs
- `xml.etree.ElementTree` — arXiv returns XML(eXtensible Markup Language), so we need this to parse it
- `anthropic` — to call Claude for summarization
- `IPython.display` — to render the markdown summary nicely in the notebook

In [1]:
import os
import requests
import xml.etree.ElementTree as ET

from dotenv import load_dotenv
from anthropic import Anthropic
from IPython.display import Markdown, display

---
## Step 2: Load API Key

We load the Anthropic API key from the `.env` file at the root of the repo.

The `load_dotenv(override=True)` call walks up the directory tree until it finds a `.env` file.

> **Note:** If you're running this for the first time, make sure you have a `.env` file in the repo root with:
> ```
> ANTHROPIC_API_KEY=sk-ant-...
> ```

In [2]:
# Load the .env file from the repo root (two levels up from this notebook)
load_dotenv(override=True)
api_key = os.getenv("ANTHROPIC_API_KEY")

# Quick sanity check — same as day1
if not api_key:
    print("No API key found. Add ANTHROPIC_API_KEY to your .env file.")
elif not api_key.startswith("sk-ant-"):
    print("API key found but doesn't look right. Double-check it starts with sk-ant-")
else:
    print("API key loaded successfully!")

API key loaded successfully!


---
## Step 3: Create the Anthropic client

We create one client object and reuse it for all our API calls.

In [3]:
# Create the Anthropic client — we'll use this throughout the notebook
anthropic = Anthropic(api_key=api_key)

---
## Step 4: Define the System Prompt

The **system prompt** tells the LLM what role it should play and how to format its response.

This is tuned specifically for research papers:
- We ask for a structured summary with specific sections
- We ask it to explain concepts in plain English (so non-experts can understand)
- We ask it to highlight why the paper matters (the "so what?")

In [4]:
system_prompt = """
You are a research assistant that specializes in making academic papers accessible to a general audience.

When given metadata and the abstract of a research paper, you will produce a structured summary with the following sections:

1. What is this paper about? — A 2-3 sentence plain-English explanation of the topic.
2. The problem they are solving — What gap or challenge does this research address?
3. Their approach — How did the authors tackle the problem?
4. Key findings — What did they discover or prove?
5. Why it matters — The real-world impact or significance of this work.
6. What are the limitations of this work? What are the future directions?
7. Who should read this? — What kind of reader or professional would benefit most.

Respond in markdown. Do not wrap the markdown in a code block — respond just with the markdown.
Keep the language clear and avoid unnecessary jargons. If technical terms are unavoidable, briefly explain them.
"""

---
## Step 5: The Summarization Function

This is the core function that calls Claude.

It takes a **text block** (the paper metadata we fetched) and returns a markdown summary.

The `user_prompt` follows the pattern: a prefix instruction + the actual content.

In [5]:
def summarize_paper(paper_text):
    """
    Takes a string containing paper metadata (title, authors, abstract, etc.)
    and returns a markdown-formatted summary from Claude.
    """
    user_prompt = (
        "Here is the metadata and abstract of a research paper.\n"
        "Please provide a structured summary as instructed.\n\n"
        + paper_text
    )

    messages = [{"role": "user", "content": user_prompt}]

    response = anthropic.messages.create(
        model="claude-sonnet-4-5-20250929",
        system=system_prompt,
        messages=messages,
        max_tokens=1000,
    )

    return response.content[0].text


def display_summary(paper_text):
    """
    Summarizes the paper and renders the markdown nicely in the notebook.
    """
    summary = summarize_paper(paper_text)
    display(Markdown(summary))

---
# Part A: arXiv API

## What is arXiv?

arXiv (pronounced "archive") is a free, open-access repository of research papers, primarily in:
- Computer Science, Machine Learning, AI
- Mathematics, Physics, Statistics
- Quantitative Biology and Economics

It's where most ML/AI papers are published **before** they appear in journals. Papers like the original **Transformer** ("Attention Is All You Need") and **GPT** papers are all on arXiv.

## Why use the arXiv API instead of scraping the website?

arXiv provides a **free, official REST API** at `https://export.arxiv.org/api/query`. 
- No API key required
- No bot detection to fight
- Returns clean, structured data in XML format
- We get exactly what we need: title, authors, abstract, categories

## How to find a paper's arXiv ID

Every arXiv paper has a URL like: `https://arxiv.org/abs/2303.08774`

The **ID** is the last part: `2303.08774`  
Format: `YYMM.NNNNN` (year, month, paper number)

We pass this ID to the API to fetch the paper's data.

## Step A1: Fetch paper data from arXiv

The arXiv API returns XML. We use Python's built-in `xml.etree.ElementTree` to parse it.

The XML structure uses **namespaces** (the `{http://...}` prefixes) — we define those upfront so our code is readable.

In [6]:
# arXiv API returns XML with these two namespaces
# We define them here so we can reference them cleanly when parsing
ARXIV_NS = {
    "atom": "http://www.w3.org/2005/Atom",       # standard Atom feed format
    "arxiv": "http://arxiv.org/schemas/atom",     # arXiv-specific extensions
}


def fetch_arxiv_paper(arxiv_id):
    """
    Fetches metadata for a paper from the arXiv API using its ID.

    Args:
        arxiv_id (str): The arXiv paper ID found in the URL, eg.: arxiv.org/abs/2303.08774

    Returns:
        str: A formatted text block with title, authors, categories, and abstract.
             Returns an error message string if the paper is not found.
    """
    # Build the API URL — we pass the paper ID as the query
    api_url = f"https://export.arxiv.org/api/query?id_list={arxiv_id}"

    print(f"Fetching from arXiv API: {api_url}")
    response = requests.get(api_url)

    # Parse the XML response
    root = ET.fromstring(response.content)

    # The API returns an Atom feed; each paper is an <entry> element
    entry = root.find("atom:entry", ARXIV_NS)

    if entry is None:
        return f"No paper found for arXiv ID: {arxiv_id}"

    # Extract the fields we care about
    title   = entry.findtext("atom:title",   default="Unknown Title",    namespaces=ARXIV_NS).strip()
    abstract = entry.findtext("atom:summary", default="No abstract available.", namespaces=ARXIV_NS).strip()

    # Authors — there can be multiple, so we collect them all into a list
    author_elements = entry.findall("atom:author", ARXIV_NS)
    authors = [el.findtext("atom:name", namespaces=ARXIV_NS) for el in author_elements]
    authors_str = ", ".join(authors) if authors else "Unknown authors"

    # Categories — arXiv uses subject codes like "cs.LG" (ML), "cs.CL" (NLP)
    category_elements = entry.findall("arxiv:primary_category", ARXIV_NS)
    categories = [el.get("term") for el in category_elements if el.get("term")]
    categories_str = ", ".join(categories) if categories else "Unknown category"

    # The canonical link to the paper page on arxiv.org
    link_element = entry.find("atom:link[@type='text/html']", ARXIV_NS)
    paper_url = link_element.get("href") if link_element is not None else f"https://arxiv.org/abs/{arxiv_id}"

    # Format everything into a clean text block for the LLM
    paper_text = (
        f"Source: arXiv\n"
        f"Paper ID: {arxiv_id}\n"
        f"URL: {paper_url}\n"
        f"Title: {title}\n"
        f"Authors: {authors_str}\n"
        f"Category: {categories_str}\n\n"
        f"Abstract:\n{abstract}"
    )

    return paper_text

## Step A2: Convenience wrapper — summarize directly from arXiv ID

This ties together the fetch + summarize steps into a single function call,
so we can just give it an arXiv ID and get a summary displayed.

In [7]:
def summarize_arxiv(arxiv_id):
    """
    Full pipeline: fetch paper from arXiv → summarize with Claude → display.

    Usage:
        summarize_arxiv("1706.03762")   # Attention Is All You Need
        summarize_arxiv("2303.08774")   # GPT-4 Technical Report
    """
    paper_text = fetch_arxiv_paper(arxiv_id)

    # Print what we fetched so we can see the raw data before summarization
    print("--- Raw paper data fetched from arXiv ---")
    print(paper_text)
    print("\n--- Sending to Claude for summarization... ---\n")

    display_summary(paper_text)

## Step A3: Try it out — "Attention Is All You Need"

Let's test with one of the most famous AI papers: **"Attention Is All You Need"** (Vaswani et al., 2017).

This is the paper that introduced the **Transformer architecture** — the foundation of all modern LLMs including GPT, Claude, and Gemini.

arXiv ID: `1706.03762`

In [8]:
summarize_arxiv("1706.03762")

Fetching from arXiv API: https://export.arxiv.org/api/query?id_list=1706.03762
--- Raw paper data fetched from arXiv ---
Source: arXiv
Paper ID: 1706.03762
URL: https://arxiv.org/abs/1706.03762v7
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Category: cs.CL

Abstract:
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 Englis

# Attention Is All You Need: A Research Summary

## 1. What is this paper about?

This paper introduces the Transformer, a revolutionary neural network architecture for processing sequences of data (like translating sentences from one language to another). Unlike previous models that processed words one at a time or used complex sliding windows, the Transformer relies entirely on "attention mechanisms"—a way for the model to focus on relevant parts of the input when producing each part of the output.

## 2. The problem they are solving

Before this work, the best language processing models used recurrent neural networks (RNNs) or convolutional neural networks (CNNs), which had significant drawbacks. RNNs had to process words sequentially (one after another), making them slow to train and unable to take advantage of modern parallel computing hardware. Both approaches were also complex and computationally expensive. The challenge was to create a simpler, faster architecture that could match or exceed their performance.

## 3. Their approach

The authors designed a model architecture based purely on attention mechanisms, eliminating recurrence and convolutions entirely. The Transformer uses "self-attention" to weigh the importance of different words in a sentence when processing each word, allowing it to capture relationships between words regardless of their distance apart. The architecture can process all words in parallel rather than sequentially, making it much faster to train. They tested this on machine translation tasks (English-to-German and English-to-French) and on parsing English sentence structure.

## 4. Key findings

The Transformer achieved state-of-the-art results on translation tasks: 28.4 BLEU score on English-to-German translation (over 2 points better than previous best) and 41.8 BLEU on English-to-French. Crucially, it trained much faster than competing models—achieving superior results in just 3.5 days on eight GPUs, a fraction of the training time required by previous approaches. The model also proved versatile, successfully generalizing to English parsing tasks with both large and limited datasets.

## 5. Why it matters

This paper fundamentally changed how we build AI systems for language and beyond. The Transformer architecture became the foundation for modern large language models like GPT, BERT, and ChatGPT. Its parallel processing capability made it possible to scale models to unprecedented sizes, enabling the current generation of AI applications in translation, content generation, code writing, and more. The simplicity and effectiveness of the attention-only approach sparked a paradigm shift across machine learning, extending beyond natural language processing to computer vision, speech recognition, and other domains.

## 6. What are the limitations of this work? What are the future directions?

While the paper demonstrates strong results, the authors note that the Transformer's quadratic complexity with sequence length (attention must compare every word to every other word) could limit its application to very long sequences. Future directions suggested include exploring local, restricted attention patterns to handle longer sequences more efficiently, and extending the model beyond text to other modalities like images and audio. The paper also opens questions about optimal model size, training strategies, and the interpretability of learned attention patterns.

## 7. Who should read this?

This paper is essential reading for machine learning researchers, NLP practitioners, and AI engineers working on language models or sequence processing tasks. Computer science students studying deep learning will find it foundational for understanding modern AI architectures. Software engineers building AI applications, data scientists working with language data, and anyone curious about the technology behind ChatGPT and similar systems will gain valuable insights into this transformative architecture.

## Step A4: Try another — GPT-4 Technical Report

Now let's try the **GPT-4 Technical Report** from OpenAI (2023).

arXiv ID: `2303.08774`

In [9]:
summarize_arxiv("2303.08774")

Fetching from arXiv API: https://export.arxiv.org/api/query?id_list=2303.08774
--- Raw paper data fetched from arXiv ---
Source: arXiv
Paper ID: 2303.08774
URL: https://arxiv.org/abs/2303.08774v6
Title: GPT-4 Technical Report
Authors:  OpenAI, Josh Achiam, Steven Adler, Sandhini Agarwal, Lama Ahmad, Ilge Akkaya, Florencia Leoni Aleman, Diogo Almeida, Janko Altenschmidt, Sam Altman, Shyamal Anadkat, Red Avila, Igor Babuschkin, Suchir Balaji, Valerie Balcom, Paul Baltescu, Haiming Bao, Mohammad Bavarian, Jeff Belgum, Irwan Bello, Jake Berdine, Gabriel Bernadett-Shapiro, Christopher Berner, Lenny Bogdonoff, Oleg Boiko, Madelaine Boyd, Anna-Luisa Brakman, Greg Brockman, Tim Brooks, Miles Brundage, Kevin Button, Trevor Cai, Rosie Campbell, Andrew Cann, Brittany Carey, Chelsea Carlson, Rory Carmichael, Brooke Chan, Che Chang, Fotis Chantzis, Derek Chen, Sully Chen, Ruby Chen, Jason Chen, Mark Chen, Ben Chess, Chester Cho, Casey Chu, Hyung Won Chung, Dave Cummings, Jeremiah Currier, Yunxing D

# GPT-4 Technical Report: Accessible Summary

## 1. What is this paper about?

This paper introduces GPT-4, an advanced AI model developed by OpenAI that can process both images and text as input and generate text responses. Unlike its predecessors, GPT-4 demonstrates performance comparable to human experts on various professional tests and academic challenges, marking a significant milestone in artificial intelligence capabilities.

## 2. The problem they are solving

The research addresses several challenges in developing large-scale AI systems: improving model performance to approach human-level competence on complex tasks, ensuring the model behaves reliably and factually, and making the training process predictable so that researchers can forecast how well their models will perform without having to train them fully first. Previous AI models struggled with consistency, accuracy, and the inability to process multiple types of input (like both text and images).

## 3. Their approach

The authors built GPT-4 using a Transformer architecture—a neural network design that excels at understanding relationships in data—and trained it to predict the next word in documents. Crucially, they developed infrastructure and methods that work consistently across different model sizes, allowing them to test smaller versions and accurately predict how a much larger model would perform. They also implemented a "post-training alignment process" to improve the model's accuracy and make it follow desired behaviors more reliably.

## 4. Key findings

GPT-4 achieved human-level performance on numerous professional and academic benchmarks, including passing a simulated bar exam with scores in the top 10% of test takers. The team successfully demonstrated that they could predict GPT-4's performance using models trained with just 1/1,000th of the computational resources, validating their scaling prediction methods and making future development more efficient and cost-effective.

## 5. Why it matters

This work represents a significant leap toward AI systems that can handle complex, real-world professional tasks. The ability to reliably predict model performance before investing massive computational resources makes AI development more sustainable and accessible. GPT-4's multimodal capabilities (processing both text and images) open new possibilities for applications in education, professional services, accessibility tools, and creative work. The improved factuality and behavioral alignment also make the technology safer and more trustworthy for practical deployment.

## 6. What are the limitations of this work? What are the future directions?

The paper notes that GPT-4 still falls short of human capability in many real-world scenarios, suggesting gaps in reasoning, reliability, and contextual understanding. The technical report intentionally omits many implementation details (like model size, training data specifics, and architectural choices), citing competitive and safety concerns, which limits reproducibility and independent verification. Future directions likely include further improving reasoning capabilities, reducing errors and "hallucinations," expanding multimodal capabilities, and developing better methods for ensuring AI safety and alignment with human values.

## 7. Who should read this?

This paper is essential reading for AI researchers and machine learning practitioners interested in large language models and scaling laws. Technology leaders, policymakers, and educators evaluating AI's potential impact on their fields would benefit from understanding GPT-4's capabilities and limitations. Additionally, professionals in law, medicine, and other fields where GPT-4 demonstrated competence should read this to understand both opportunities and risks. Finally, anyone concerned with AI safety and responsible deployment will find valuable insights into alignment approaches.

---
# Part B: Semantic Scholar API

## What is Semantic Scholar?

Semantic Scholar is a **free academic search engine** built by the Allen Institute for AI.
It indexes over **200 million papers** across all fields — not just CS/ML like arXiv.

This makes it useful for:
- Papers in medicine, biology, economics, social sciences
- Papers that are NOT on arXiv
- Getting extra metadata like **citation counts** and **references**

## How to find a paper's Semantic Scholar ID

You have two options:

**Option 1 — Search by title:**  
Use the `/paper/search` endpoint with the paper title.

**Option 2 — Use the arXiv ID directly:**  
Semantic Scholar understands arXiv IDs! You can pass `ArXiv:1706.03762` as the paper ID.

**Option 3 — Use the Semantic Scholar paper URL:**  
URLs look like: `https://www.semanticscholar.org/paper/Attention-is-all-you-need/204e3073870fae3d05bcbc2f6a8e263d9b72e776`  
The ID is the long hash at the end: `204e3073870fae3d05bcbc2f6a8e263d9b72e776`

## API rate limits

The free tier allows **100 requests per 5 minutes** — more than enough for our use case.
No API key is required for basic usage.

## Step B1: Fetch paper data from Semantic Scholar

The Semantic Scholar API returns clean JSON (much simpler than arXiv's XML).

We request specific **fields** in the API call so we only get what we need.

In [10]:
# Base URL for the Semantic Scholar API
S2_BASE_URL = "https://api.semanticscholar.org/graph/v1"

# The fields we want from the API
# Requesting only what we need keeps responses fast and avoids hitting limits
S2_FIELDS = "title,authors,year,abstract,citationCount,fieldsOfStudy,externalIds"


def fetch_semantic_scholar_paper(paper_id):
    """
    Fetches metadata for a paper from the Semantic Scholar API.

    Args:
        paper_id (str): Can be any of:
            - Semantic Scholar hash ID: "204e3073870fae3d05bcbc2f6a8e263d9b72e776"
            - arXiv ID with prefix:     "ArXiv:1706.03762"
            - DOI with prefix:          "DOI:10.1145/3442188.3445922"

    Returns:
        str: A formatted text block with paper metadata, or an error message.
    """
    api_url = f"{S2_BASE_URL}/paper/{paper_id}?fields={S2_FIELDS}"

    print(f"Fetching from Semantic Scholar API: {api_url}")
    response = requests.get(api_url)

    # Check if the request succeeded (HTTP 200 = OK)
    if response.status_code != 200:
        return f"Error {response.status_code}: Could not fetch paper '{paper_id}'. Check that the ID is correct."

    data = response.json()

    # Extract fields — .get() with a default value handles missing fields safely
    title          = data.get("title", "Unknown Title")
    year           = data.get("year", "Unknown Year")
    abstract       = data.get("abstract") or "No abstract available."
    citation_count = data.get("citationCount", 0)
    fields         = data.get("fieldsOfStudy") or []
    fields_str     = ", ".join(fields) if fields else "Unknown"

    # Authors are returned as a list of objects with a 'name' key
    authors_list = data.get("authors", [])
    authors_str  = ", ".join(a["name"] for a in authors_list) if authors_list else "Unknown authors"

    # Build the URL for the paper's Semantic Scholar page
    s2_id    = data.get("paperId", "")
    paper_url = f"https://www.semanticscholar.org/paper/{s2_id}" if s2_id else "URL not available"

    # Format into a clean text block for the LLM
    paper_text = (
        f"Source: Semantic Scholar\n"
        f"Paper ID: {paper_id}\n"
        f"URL: {paper_url}\n"
        f"Title: {title}\n"
        f"Authors: {authors_str}\n"
        f"Year: {year}\n"
        f"Fields of Study: {fields_str}\n"
        f"Citation Count: {citation_count:,}\n\n"
        f"Abstract:\n{abstract}"
    )

    return paper_text

## Step B2: Convenience wrapper — summarize from Semantic Scholar

Same pattern as the arXiv wrapper above.

In [11]:
def summarize_semantic_scholar(paper_id):
    """
    Full pipeline: fetch paper from Semantic Scholar → summarize with Claude → display.

    Usage:
        # Using arXiv ID prefix (easiest if you know the arXiv ID):
        summarize_semantic_scholar("ArXiv:1706.03762")

        # Using Semantic Scholar's own hash ID:
        summarize_semantic_scholar("204e3073870fae3d05bcbc2f6a8e263d9b72e776")

        # Using a DOI:
        summarize_semantic_scholar("DOI:10.1038/s41586-021-03819-2")
    """
    paper_text = fetch_semantic_scholar_paper(paper_id)

    print("--- Raw paper data fetched from Semantic Scholar ---")
    print(paper_text)
    print("\n--- Sending to Claude for summarization... ---\n")

    display_summary(paper_text)

## Step B3: Try it — same paper via Semantic Scholar

Let's fetch the same "Attention Is All You Need" paper, but this time through Semantic Scholar.

We can reuse the arXiv ID by adding the `ArXiv:` prefix — Semantic Scholar understands it.

In [12]:
# Note: Semantic Scholar adds extra info like citation count and fields of study
# that arXiv doesn't provide — so the summary may have richer context
summarize_semantic_scholar("ArXiv:1706.03762")

Fetching from Semantic Scholar API: https://api.semanticscholar.org/graph/v1/paper/ArXiv:1706.03762?fields=title,authors,year,abstract,citationCount,fieldsOfStudy,externalIds
--- Raw paper data fetched from Semantic Scholar ---
Source: Semantic Scholar
Paper ID: ArXiv:1706.03762
URL: https://www.semanticscholar.org/paper/204e3073870fae3d05bcbc2f6a8e263d9b72e776
Title: Attention is All you Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, I. Polosukhin
Year: 2017
Fields of Study: Computer Science
Citation Count: 170,825

Abstract:
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Ex

# Attention is All You Need: A Research Summary

## What is this paper about?

This paper introduces the **Transformer**, a revolutionary neural network architecture for processing sequences of data (like translating text from one language to another). Unlike previous models that processed information step-by-step, the Transformer uses only "attention mechanisms"—a way for the model to focus on relevant parts of the input—making it faster and more efficient.

## The problem they are solving

Before this work, the best models for sequence tasks like translation relied on recurrent neural networks (RNNs) or convolutional neural networks (CNNs), which processed data sequentially or in fixed windows. This made them slow to train because computations couldn't be easily parallelized (run simultaneously). The challenge was to build a model that could match or exceed their performance while being faster and more efficient to train.

## Their approach

The authors designed the Transformer architecture, which eliminates recurrence and convolutions entirely. Instead, it uses **self-attention mechanisms** that allow the model to weigh the importance of different words in a sentence relative to each other, regardless of their distance apart. This design enables parallel processing of all words simultaneously, dramatically speeding up training. The architecture consists of stacked encoder and decoder layers, each using multi-head attention and feed-forward networks.

## Key findings

- The Transformer achieved state-of-the-art results on machine translation tasks, scoring 28.4 BLEU on English-to-German translation (a 2+ point improvement) and 41.8 BLEU on English-to-French translation
- Training was significantly faster: the model reached top performance in just 3.5 days on 8 GPUs, while competing models required much more computational resources
- The architecture proved versatile, successfully generalizing to other tasks like English constituency parsing (analyzing sentence structure)
- The model was more parallelizable, making better use of modern computing hardware

## Why it matters

This paper fundamentally transformed artificial intelligence and natural language processing. The Transformer architecture became the foundation for virtually all modern large language models, including BERT, GPT, and ChatGPT. Its efficiency and effectiveness enabled the AI revolution we're experiencing today, making it possible to train increasingly powerful models that can understand and generate human language, translate between languages, write code, and perform countless other tasks. With over 170,000 citations, it's one of the most influential AI papers ever published.

## What are the limitations of this work? What are the future directions?

While not explicitly detailed in the abstract, the original Transformer still required substantial computational resources for training. Future directions suggested by this work include scaling to even larger models, applying the architecture to domains beyond text (like images and audio), and optimizing efficiency further. Indeed, subsequent research has explored these directions, leading to multimodal transformers, more efficient attention mechanisms, and architectural refinements that continue to push the boundaries of what's possible.

## Who should read this?

- **AI researchers and machine learning engineers** working on natural language processing, computer vision, or any sequence modeling task
- **Computer science students** seeking to understand foundational concepts in modern deep learning
- **Tech professionals and product managers** who want to understand the technology behind ChatGPT and similar AI systems
- **Anyone curious about AI** who wants to understand the breakthrough that enabled today's generative AI revolution

## Step B4: Try a paper outside CS — a biology/nature paper

Semantic Scholar shines for non-CS papers. Let's try the **AlphaFold 2** paper (protein structure prediction).

This paper is from *Nature* and is NOT on arXiv — you can only access it via Semantic Scholar or DOI.

DOI: `10.1038/s41586-021-03819-2`

In [13]:
summarize_semantic_scholar("DOI:10.1038/s41586-021-03819-2")

Fetching from Semantic Scholar API: https://api.semanticscholar.org/graph/v1/paper/DOI:10.1038/s41586-021-03819-2?fields=title,authors,year,abstract,citationCount,fieldsOfStudy,externalIds
--- Raw paper data fetched from Semantic Scholar ---
Source: Semantic Scholar
Paper ID: DOI:10.1038/s41586-021-03819-2
URL: https://www.semanticscholar.org/paper/dc32a984b651256a8ec282be52310e6bd33d9815
Title: Highly accurate protein structure prediction with AlphaFold
Authors: J. Jumper, Richard Evans, A. Pritzel, Tim Green, Michael Figurnov, O. Ronneberger, Kathryn Tunyasuvunakool, Russ Bates, Augustin Žídek, Anna Potapenko, Alex Bridgland, Clemens Meyer, Simon A A Kohl, Andy Ballard, A. Cowie, Bernardino Romera-Paredes, Stanislav Nikolov, Rishub Jain, J. Adler, T. Back, Stig Petersen, D. Reiman, Ellen Clancy, Michal Zielinski, Martin Steinegger, Michalina Pacholska, Tamas Berghammer, Sebastian Bodenstein, David Silver, O. Vinyals, A. Senior, K. Kavukcuoglu, Pushmeet Kohli, D. Hassabis
Year: 2021
F

# Highly Accurate Protein Structure Prediction with AlphaFold

## What is this paper about?

This paper introduces AlphaFold, a groundbreaking artificial intelligence system that can predict the 3D structure of proteins from their amino acid sequences with accuracy comparable to experimental methods. This addresses one of biology's longest-standing challenges: the protein folding problem.

## The problem they are solving

While proteins are fundamental to all life processes and knowing their 3D structure is crucial for understanding how they work, experimental methods to determine protein structures are extremely slow and labor-intensive—taking months to years per protein. Out of billions of known protein sequences, only about 100,000 structures had been experimentally determined. Previous computational prediction methods were far less accurate than experiments, especially when no similar protein structure was already known.

## Their approach

The researchers developed AlphaFold using a novel deep learning architecture that combines:
- Neural networks trained on existing protein structure data
- Physical and biological knowledge about how proteins fold
- Multi-sequence alignments (comparing related protein sequences across different organisms)
- An innovative approach that integrates these elements into the AI's design rather than treating them as separate components

The system was validated in CASP14, a rigorous blind competition where researchers predict structures of proteins whose experimental structures are known but not yet publicly released.

## Key findings

AlphaFold achieved accuracy competitive with experimental methods (like X-ray crystallography or cryo-electron microscopy) in the majority of test cases. This represents the first time a computational method has reached "atomic accuracy"—meaning predictions are precise enough to be useful for understanding protein function and drug design. The system dramatically outperformed all other computational methods and succeeded even when no similar protein structures were available as templates.

## Why it matters

This breakthrough has transformative implications for biology and medicine:
- **Drug discovery**: Understanding protein structures helps identify targets for new medications
- **Disease understanding**: Many diseases involve misfolded proteins
- **Biotechnology**: Enables design of new proteins for industrial or therapeutic purposes
- **Research acceleration**: What once took months can now be done in hours, potentially unlocking structural information for billions of proteins

The work democratizes access to structural biology, making it available to researchers without access to expensive experimental facilities.

## What are the limitations of this work? What are the future directions?

While not explicitly detailed in this abstract, typical limitations would include:
- Accuracy varies across different protein types
- Computational resources still required for predictions
- Some complex cases (like proteins with multiple conformations or those involving interactions with other molecules) remain challenging
- Need for continued validation across diverse protein families

Future directions likely include extending predictions to protein complexes, protein-drug interactions, and understanding dynamic protein movements.

## Who should read this?

- **Structural biologists** seeking to accelerate their research
- **Drug discovery scientists** working on therapeutic design
- **Computational biologists** interested in AI applications
- **Biochemists and molecular biologists** studying protein function
- **AI researchers** interested in scientific applications of deep learning
- **Medical researchers** working on diseases involving protein dysfunction

This paper is foundational reading for anyone working at the intersection of biology, medicine, and artificial intelligence.

---
# Part C: Search by Title (Semantic Scholar)

So far we've been fetching papers when we already know the ID.
But what if you only know the **title** of a paper?

Semantic Scholar has a **search endpoint** that lets us search by keyword or title.

This is handy when:
- You read a paper title in a blog post and want to summarize it quickly
- You want to explore papers on a topic without knowing specific IDs

## A note on the API endpoint

Semantic Scholar has two search endpoints:

| Endpoint | Rate limit | Sort support |
|---|---|---|
| `/paper/search` | Very aggressive — hits 429 quickly | Relevance only |
| `/paper/search/bulk` | Much more permissive | Supports `citationCount:desc` |

We use `/paper/search/bulk` and sort by **citation count descending**.
This means the most well-known paper for a given query comes first — which is usually what you want.

In [17]:
def search_and_summarize(query, limit=3):
    """
    Searches Semantic Scholar for papers matching the query,
    shows the top results sorted by citation count, then summarizes the top match.

    Args:
        query (str): A paper title or keyword search
        limit (int): How many search results to show before picking the top one.
    """
    # Build the search URL using the bulk endpoint
    # sort=citationCount:desc puts the most-cited (usually most relevant) paper first
    search_url = (
        f"{S2_BASE_URL}/paper/search/bulk"
        f"?query={requests.utils.quote(query)}"
        f"&limit={limit}"
        f"&fields=paperId,title,year,citationCount,authors"
        f"&sort=citationCount:desc"
    )

    print(f"Searching Semantic Scholar for: '{query}'")
    response = requests.get(search_url)

    if response.status_code != 200:
        print(f"Search failed with status {response.status_code}: {response.text[:200]}")
        return

    results = response.json().get("data", [])

    if not results:
        print("No papers found for that query.")
        return

    # Display the search results so the user can see what was found
    print(f"\nTop {len(results)} results (sorted by citation count):")
    print("-" * 60)
    for i, paper in enumerate(results, start=1):
        authors = paper.get("authors", [])
        first_author = authors[0]["name"] if authors else "Unknown"
        print(f"{i}. {paper['title']}")
        print(f"   By: {first_author} et al. | Year: {paper.get('year', 'N/A')} | Citations: {paper.get('citationCount', 0):,}")
        print()

    # Automatically pick the top result (most cited) and summarize it
    top_paper_id = results[0]["paperId"]
    print(f"Summarizing top result (paper ID: {top_paper_id})...")
    print("=" * 60 + "\n")

    summarize_semantic_scholar(top_paper_id)

## Step C1: Search by topic — let's try "BERT"

BERT (Bidirectional Encoder Representations from Transformers) is a hugely influential NLP model from Google.

In [18]:
search_and_summarize("BERT pre-training language model")

Searching Semantic Scholar for: 'BERT pre-training language model'

Top 1000 results (sorted by citation count):
------------------------------------------------------------
1. BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
   By: Jacob Devlin et al. | Year: 2019 | Citations: 112,088

2. BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension
   By: M. Lewis et al. | Year: 2019 | Citations: 12,477

3. DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter
   By: Victor Sanh et al. | Year: 2019 | Citations: 9,361

4. BioBERT: a pre-trained biomedical language representation model for biomedical text mining
   By: Jinhyuk Lee et al. | Year: 2019 | Citations: 7,027

5. HuBERT: Self-Supervised Speech Representation Learning by Masked Prediction of Hidden Units
   By: Wei-Ning Hsu et al. | Year: 2021 | Citations: 4,310

6. DeBERTa: Decoding-enhanced BERT with Disentangled Atten

# BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding

## 1. What is this paper about?

This paper introduces BERT (Bidirectional Encoder Representations from Transformers), a revolutionary language model that learns to understand text by reading it in both directions simultaneously—left-to-right and right-to-left. Unlike previous models that only read text in one direction, BERT can better understand context and word meanings by considering the full surrounding text.

## 2. The problem they are solving

Before BERT, language models typically processed text in only one direction (left-to-right), which limited their ability to understand context. For example, in the sentence "I went to the bank to deposit money," these models struggled to understand that "bank" refers to a financial institution rather than a riverbank because they couldn't look ahead at "deposit money." This unidirectional approach meant researchers had to build custom models for each specific language task, which was time-consuming and often less effective.

## 3. Their approach

The authors developed a two-stage approach:
- **Pre-training**: They trained BERT on massive amounts of unlabeled text using two clever tasks: (1) predicting randomly masked words in sentences, and (2) determining if two sentences naturally follow each other. This forces the model to learn deep language understanding by reading context from both directions.
- **Fine-tuning**: The pre-trained BERT can then be adapted to specific tasks (like question answering or sentiment analysis) by adding just one simple output layer and training briefly on task-specific data.

## 4. Key findings

BERT achieved groundbreaking results, setting new records on 11 different language understanding tasks. Notable improvements include:
- 7.7-point improvement on the GLUE benchmark (a collection of diverse language tasks)
- 4.6% improvement on natural language inference tasks
- Significant gains on question-answering tasks (SQuAD v1.1 and v2.0)

The model proved that bidirectional understanding is crucial—the authors showed that reading context from both directions substantially outperforms unidirectional approaches.

## 5. Why it matters

BERT fundamentally changed how we build AI language systems. Instead of creating specialized models for each task from scratch, practitioners can now start with pre-trained BERT and quickly adapt it to their needs—saving time, computational resources, and often achieving better results. This approach has become the foundation for modern NLP applications like search engines, chatbots, translation services, and voice assistants. BERT's success democratized access to powerful language AI, making it feasible for smaller organizations to build sophisticated language applications.

## 6. What are the limitations of this work? What are the future directions?

**Limitations:**
- BERT requires substantial computational resources for pre-training (trained on multiple GPUs/TPUs for days)
- The model is large and can be slow for real-time applications
- Pre-training only uses unlabeled text; it doesn't directly learn from structured knowledge
- The masking approach during training creates a mismatch with how the model is used during actual application

**Future directions:**
- Creating more efficient versions that maintain performance with fewer parameters
- Developing better pre-training objectives that more closely match downstream tasks
- Incorporating structured knowledge and multi-modal information (images, audio)
- Extending to longer documents (BERT was limited to 512 tokens)
- Improving cross-lingual capabilities

## 7. Who should read this?

- **NLP researchers and practitioners** wanting to understand the foundation of modern language models
- **Machine learning engineers** building language-based applications
- **Data scientists** working with text analysis, sentiment analysis, or information extraction
- **Product managers and tech leaders** making decisions about AI capabilities in their products
- **Computer science students** learning about deep learning and natural language processing
- Anyone interested in how AI systems like ChatGPT, search engines, or virtual assistants understand language

---
# Quick Reference: How to Use This Notebook

## If you have an arXiv ID
```python
# Go to arxiv.org, find your paper — the ID is in the URL
# e.g. arxiv.org/abs/2310.06825  →  ID is "2310.06825"
summarize_arxiv("2310.06825")
```

## If you have a paper title or keyword
```python
search_and_summarize("retrieval augmented generation")
search_and_summarize("diffusion models image generation")
```

## If you have a DOI
```python
# DOIs look like: 10.1038/s41586-021-03819-2
summarize_semantic_scholar("DOI:10.1038/s41586-021-03819-2")
```

## Some interesting papers to try

| Paper | arXiv ID |
|---|---|
| Attention Is All You Need (Transformers) | `1706.03762` |
| GPT-4 Technical Report | `2303.08774` |
| DALL-E 2 | `2204.06125` |
| Llama 2 | `2307.09288` |
| Retrieval-Augmented Generation (RAG) | `2005.11401` |
| LoRA (low-rank fine-tuning) | `2106.09685` |

In [ ]:
# Your turn — paste any arXiv ID or search query here!
# summarize_arxiv("YOUR_ARXIV_ID_HERE")
# search_and_summarize("YOUR SEARCH QUERY HERE")